# Práctica 2 — Titanic Dataset
**Nombre:** Camilo Abalco
**Materia:** Introducción a los Sistemas de Información
**Fecha:** 13 de noviembre de 2026

In [ ]:
import pandas as pd

pd.set_option("display.width", 120)

df = pd.read_csv("titanic.csv")

print(df.shape)
print(df.isna().sum().to_frame("Valores faltantes"))

## Pregunta 1
¿Cuántos pasajeros hay en el dataset y cuántos PassengerId distintos aparecen? ¿Coinciden los dos resultados?

In [ ]:
total_registros = len(df)
ids_distintos = df["PassengerId"].nunique()

tabla1 = pd.DataFrame({
    "Medida": ["Registros (filas)", "PassengerId distintos", "¿Coinciden?"],
    "Valor": [total_registros, ids_distintos, total_registros == ids_distintos]
})
print(tabla1.to_string(index=False))

**Interpretación:** El archivo contiene 891 registros y 891 PassengerId distintos, por lo que coinciden: cada fila corresponde a un pasajero con identificador único.

## Pregunta 2
¿Cuántos pasajeros de cada clase aparecen en el archivo? ¿Qué clase tiene más pasajeros registrados?

In [ ]:
tabla2 = (df["Pclass"].value_counts().sort_index()
          .rename_axis("Clase").reset_index(name="Pasajeros"))
tabla2["Clase"] = tabla2["Clase"].map({1: "Primera", 2: "Segunda", 3: "Tercera"})

print(tabla2.to_string(index=False))
print("\nClase con más pasajeros:", tabla2.loc[tabla2["Pasajeros"].idxmax(), "Clase"])

assert tabla2["Pasajeros"].sum() == len(df)

**Interpretación:** En el archivo hay 216 pasajeros de primera clase, 184 de segunda y 491 de tercera. La clase con más pasajeros registrados es la tercera. Los subtotales suman 891, igual que el total del archivo.

## Pregunta 3
¿Cuántos pasajeros embarcaron en cada puerto y cuántos tienen el puerto sin registrar?

In [ ]:
puertos = {"C": "Cherburgo", "Q": "Queenstown", "S": "Southampton"}

tabla3 = (df["Embarked"].map(puertos).fillna("Sin registrar")
          .value_counts().rename_axis("Puerto").reset_index(name="Pasajeros"))
tabla3.loc[len(tabla3)] = ["TOTAL", tabla3["Pasajeros"].sum()]

print(tabla3.to_string(index=False))

assert tabla3.iloc[-1]["Pasajeros"] == len(df)

**Interpretación:** En el archivo, 644 pasajeros embarcaron en Southampton, 168 en Cherburgo y 77 en Queenstown. Hay 2 registros sin puerto. Los valores faltantes no se eliminaron: se agruparon como "Sin registrar" para que el total siga siendo 891 (644 + 168 + 77 + 2).

## Pregunta 4
¿Cuál es la tarifa promedio, mínima y máxima registrada para cada clase?

In [ ]:
tabla4 = (df.groupby("Pclass")["Fare"]
          .agg(Promedio="mean", Mínima="min", Máxima="max", Registros_con_tarifa="count")
          .round(2).reset_index())
tabla4["Pclass"] = tabla4["Pclass"].map({1: "Primera", 2: "Segunda", 3: "Tercera"})
tabla4 = tabla4.rename(columns={"Pclass": "Clase"})

print(tabla4.to_string(index=False))
print("\nTarifas faltantes:", df["Fare"].isna().sum())

**Interpretación:** La tarifa promedio registrada fue de 84.15 en primera, 20.66 en segunda y 13.68 en tercera. La tarifa mínima es 0.00 en las tres clases, y la máxima es 512.33 en primera, 73.50 en segunda y 69.55 en tercera. No hay valores faltantes en `Fare` (0 en el archivo), por lo que los cálculos usan los 891 registros. Cabe notar que existen tarifas de 0 y que un boleto compartido puede repetir su tarifa en varias filas.

## Pregunta 5
¿Cuántos pasajeros de cada clase embarcaron en cada puerto?

In [ ]:
df["Puerto"] = df["Embarked"].map(puertos).fillna("Sin registrar")
df["Clase"] = df["Pclass"].map({1: "Primera", 2: "Segunda", 3: "Tercera"})

tabla5 = pd.crosstab(df["Clase"], df["Puerto"], margins=True, margins_name="Total")
print(tabla5)

assert tabla5.loc["Total", "Total"] == len(df)

**Interpretación:** En Southampton embarcaron 127 pasajeros de primera, 164 de segunda y 353 de tercera. En Cherburgo, 85 de primera, 17 de segunda y 66 de tercera. En Queenstown, 2 de primera, 3 de segunda y 72 de tercera. Los 2 registros sin puerto pertenecen a primera clase y aparecen en su propia columna. El total es 891.

## Pregunta 6
¿Cuántos pasajeros sobrevivieron y cuántos fallecieron en cada clase?

In [ ]:
df["Resultado"] = df["Survived"].map({0: "Falleció", 1: "Sobrevivió"})

tabla6 = pd.crosstab(df["Clase"], df["Resultado"], margins=True, margins_name="Total")
tabla6["% Sobrevivió"] = (tabla6["Sobrevivió"] / tabla6["Total"] * 100).round(1)
print(tabla6)

assert tabla6.loc["Total", "Total"] == len(df)

In [ ]:
**Interpretación:** En el archivo, sobrevivieron 342 pasajeros y fallecieron 549. Por clase, sobrevivió el 63.0% en primera (136 de 216), el 47.3% en segunda (87 de 184) y el 24.2% en tercera (119 de 491). La clase con mayor proporción de sobrevivientes es la primera. Los subtotales suman 891.

## Pregunta 7
¿Cuántos pasajeros viajaban con cero, uno o dos o más familiares registrados a bordo?

In [ ]:
df["Familiares"] = df["SibSp"] + df["Parch"]

df["Grupo familiar"] = pd.cut(
    df["Familiares"], bins=[-1, 0, 1, float("inf")],
    labels=["0 familiares", "1 familiar", "2 o más familiares"]
)

tabla7 = (df["Grupo familiar"].value_counts().sort_index()
          .rename_axis("Familiares a bordo").reset_index(name="Pasajeros"))
tabla7.loc[len(tabla7)] = ["TOTAL", tabla7["Pasajeros"].sum()]

print(tabla7.to_string(index=False))

assert tabla7.iloc[-1]["Pasajeros"] == len(df)

**Interpretación:** Viajaban sin familiares registrados 537 pasajeros, con un familiar 161 y con dos o más 193. Los familiares se calcularon sumando `SibSp` y `Parch`, columnas sin valores faltantes. El total es 891 (537 + 161 + 193).

## Pregunta 8
¿Cuántos pasajeros tenían menos de 18 años, cuántos tenían 18 o más y cuántos tenían la edad sin registrar? Desglose el resultado por clase.

In [ ]:
def grupo_edad(edad):
    if pd.isna(edad):
        return "Edad sin registrar"
    return "Menor de 18" if edad < 18 else "18 o más"

df["Grupo de edad"] = df["Age"].apply(grupo_edad)

tabla8 = pd.crosstab(df["Clase"], df["Grupo de edad"], margins=True, margins_name="Total")
tabla8 = tabla8[["Menor de 18", "18 o más", "Edad sin registrar", "Total"]]
print(tabla8)

assert tabla8.loc["Total", "Total"] == len(df)

**Interpretación:** En el archivo hay 113 pasajeros menores de 18 años, 601 de 18 o más y 177 sin edad registrada. Por clase, los menores de 18 son 14 en primera, 24 en segunda y 75 en tercera; los de 18 o más, 172, 149 y 280; y los sin edad, 30, 11 y 136. La clase con más registros sin edad es la tercera. Los pasajeros sin edad no se descartaron ni se estimaron: se muestran en su propia columna para que los subtotales sumen 891.

## Pregunta 9
¿Cuántos códigos de boleto distintos hay? ¿Cuántos aparecen asociados a más de un pasajero y cuál aparece en más registros?

In [ ]:
conteo_boletos = df["Ticket"].value_counts()

distintos = df["Ticket"].nunique()
compartidos = (conteo_boletos > 1).sum()
max_registros = conteo_boletos.max()
boletos_max = conteo_boletos[conteo_boletos == max_registros].index.tolist()

tabla9 = pd.DataFrame({
    "Medida": ["Códigos de boleto distintos",
               "Códigos asociados a más de un pasajero",
               "Máximo de registros para un mismo código",
               "Código(s) con más registros"],
    "Valor": [distintos, compartidos, max_registros, ", ".join(boletos_max)]
})
print(tabla9.to_string(index=False))

assert conteo_boletos.sum() == len(df)

**Interpretación:** Hay 681 códigos de boleto distintos en el archivo. De ellos, 134 aparecen asociados a más de un pasajero. El máximo es de 7 registros por código, y hay tres códigos empatados con ese máximo: 347082, CA. 2343 y 1601. Por eso la tabla lista los tres.

## Pregunta 10
¿En qué clase hay más registros sin cabina? Para cada clase, muestren cuántos registros tienen cabina, cuántos carecen de ella y cuántos hay en total.

In [ ]:
df["Cabina"] = df["Cabin"].notna().map({True: "Con cabina", False: "Sin cabina"})

tabla10 = pd.crosstab(df["Clase"], df["Cabina"], margins=True, margins_name="Total")
tabla10 = tabla10[["Con cabina", "Sin cabina", "Total"]]
print(tabla10)

clases_sin_cab = tabla10.drop("Total")["Sin cabina"]
print("\nClase con más registros sin cabina:", clases_sin_cab.idxmax())

assert tabla10.loc["Total", "Total"] == len(df)

**Interpretación:** La clase con más registros sin cabina es la tercera, con 479 registros. En primera hay 176 con cabina y 40 sin ella; en segunda, 16 y 168; en tercera, 12 y 479. En total, 204 registros tienen cabina y 687 no. Un valor vacío en `Cabin` significa que la cabina no figura en el registro, no necesariamente que el pasajero no tuviera una.

## Tratamiento de valores faltantes

| Columna | Tratamiento |
|---|---|
| `Embarked` | Se agrupó como "Sin registrar" (Preguntas 3 y 5). |
| `Age` | Se agrupó como "Edad sin registrar" (Pregunta 8). |
| `Cabin` | Se interpretó como "Sin cabina registrada" (Pregunta 10). |
| `Fare` | Los cálculos ignoran los valores faltantes y se reporta cuántos hay (Pregunta 4). |

## Conclusión
En todas las tablas, los subtotales coinciden con los 891 registros del archivo. Los resultados describen únicamente a los pasajeros incluidos en este archivo, no a todos los pasajeros del Titanic. La tercera clase concentra más de la mitad de los registros (491 de 891) y tiene la menor proporción de sobrevivientes (24.2%), mientras que la primera clase tiene la mayor (63.0%). Además, la información de cabina y edad está muy incompleta, sobre todo en tercera clase, lo que limita lo que se puede concluir con esas columnas.